# Bronze to Silver Full Load
Reads every bronze table, trims string columns, writes an idempotent full overwrite to silver, then verifies row counts.

In [0]:
# Cell 1 — Configuration
table_names = [
    "product",
    "purchase_order_detail",
    "purchase_order_header",
    "sales_order_detail",
    "sales_territory",
    "vendor",
]
source_schema = "adventure_works.bronze"
target_schema = "adventure_works.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema}")

DataFrame[]

In [0]:
# Cell 2 — Load with light cleaning
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

for t in table_names:
    df = spark.table(f"{source_schema}.{t}")
    cleaned = df.select([
        F.trim(F.col(f.name)).alias(f.name) if isinstance(f.dataType, StringType) else F.col(f.name)
        for f in df.schema.fields
    ])
    (cleaned.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{target_schema}.{t}"))
    print(f"{t}: {spark.table(f'{target_schema}.{t}').count():,} rows written")

product: 504 rows written
purchase_order_detail: 8,845 rows written
purchase_order_header: 4,012 rows written
sales_order_detail: 121,317 rows written
sales_territory: 10 rows written
vendor: 104 rows written


In [0]:
# Cell 3 — Verification
rows = []
for t in table_names:
    b = spark.table(f"{source_schema}.{t}").count()
    s = spark.table(f"{target_schema}.{t}").count()
    rows.append((t, b, s, b == s))

result = spark.createDataFrame(rows, "table string, bronze_rows long, silver_rows long, counts_match boolean")
display(result)

assert all(r[3] for r in rows), "Row count mismatch between bronze and silver — check Cell 2"
print("All tables loaded successfully: bronze and silver row counts match.")

table,bronze_rows,silver_rows,counts_match
product,504,504,true
purchase_order_detail,8845,8845,true
purchase_order_header,4012,4012,true
sales_order_detail,121317,121317,true
sales_territory,10,10,true
vendor,104,104,true


All tables loaded successfully: bronze and silver row counts match.
